# Tammes Problem (n = 50) — Analysis & Verification

This notebook verifies solutions for the **Tammes problem** (Einstein Arena slug `tammes-problem`).

**Problem:** Place 50 points on the unit sphere to maximize the minimum pairwise Euclidean distance
$$d_{\min} = \min_{1 \le i < j \le 50} \|\mathbf{p}_i - \mathbf{p}_j\|.$$
Points are projected onto $S^2$ before scoring. **Higher is better.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, "solutions")

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

## 1. Verification function

Exact **Einstein Arena** verifier (copied from `einstein-arena/web/src/lib/problems/tammes-problem.ts`):

In [ ]:
def evaluate(data):
    vectors = np.array(data["vectors"], dtype=np.float64)
    assert vectors.shape == (50, 3), f"Expected (50, 3), got {vectors.shape}"
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms < 1e-12] = 1e-12
    vectors = vectors / norms
    diffs = vectors[:, None, :] - vectors[None, :, :]
    dist_sq = np.sum(diffs**2, axis=2)
    iu = np.triu_indices(50, k=1)
    dists = np.sqrt(dist_sq[iu])
    return float(np.min(dists))

## 2. Load solutions

In [ ]:
from alphaevolve_2025 import vectors as v_ae
from ours_2026 import vectors as v_ours

payloads = [
    ("AlphaEvolve V2 (baseline)", {"vectors": v_ae.tolist()}),
    ("Ours (2026)", {"vectors": v_ours.tolist()}),
]

for name, data in payloads:
    v = np.array(data["vectors"], dtype=np.float64)
    print(f"{name}: {v.shape[0]} points in R^3")

## 3. Verify score

In [ ]:
print("=" * 70)
print("VERIFICATION (Einstein Arena `evaluate`)")
print("=" * 70)

for name, data in payloads:
    s = evaluate(data)
    print(f"{name}: d_min = {s:.15f}")

print()
print("Higher d_min is better.")

## 4. Visualization

3D scatter of the 50 points on the unit sphere for each configuration.

In [ ]:
colors = ["#C850C0", "#E67E22"]

fig = plt.figure(figsize=(12, 5))
for idx, ((name, data), color) in enumerate(zip(payloads, colors)):
    v = np.array(data["vectors"], dtype=np.float64)
    norms = np.linalg.norm(v, axis=1, keepdims=True)
    norms[norms < 1e-12] = 1e-12
    v = v / norms
    ax = fig.add_subplot(1, 2, idx + 1, projection="3d")
    ax.scatter(v[:, 0], v[:, 1], v[:, 2], s=30, color=color, edgecolors="k", linewidths=0.3)
    ax.set_title(name)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_zlim(-1, 1)
    ax.set_box_aspect([1, 1, 1])
plt.tight_layout()
plt.show()